# `inference.py` — SentimentPredictor: Model-in-Production Wrapper

## Purpose
Wraps the trained `MultiAspectSentimentModel` for single-review inference at runtime. This is the **core production module** — both `trained_model_adapter.py` (website API) and `trained_model_xai.py` (explainability) use `SentimentPredictor` internally.

## Key Design Decisions

### 1. Empty edge_index forces the correct model path
The model has two forward paths:
- `edge_index=None` → short-circuits to raw `attn_predictions` from `AspectAwareRoBERTa`, **bypassing** the `final_classifier`
- `edge_index=[empty_tensor]` → enters the full GCN branch (falls back to zero GCN output), then applies `final_classifier`

The `final_classifier` is what was actually optimised during training (it maps `[attention + GCN]` → `logits`). Inference MUST pass an empty edge_index to use the correct code path.

### 2. Temperature scaling replaces Platt scaling
The model's raw logits are in a small range (~0.1–0.3), so softmax outputs are near-uniform (~0.33 each). Dividing by `temperature=0.5` amplifies the logit differences, producing confident predictions without retraining or a separate calibration step.

### 3. Inference-time text cleaning
The same garbled-text and translation-artifact cleaning pipeline used during preprocessing is applied to user-submitted text. This ensures the input distribution matches what the model was trained on.

## Available Explanation Methods
| Method | Speed | Type |
|--------|-------|------|
| `predict(return_attention=True)` | Fast | MHA attention weights |
| `explain_with_integrated_gradients()` | Medium | Gradient-based attribution |
| `explain_with_lime()` | Slow | Model-agnostic perturbation |
| `explain_with_shap()` | Slow | Shapley values |

In [1]:
from tqdm.auto import tqdm
print("⏳ Starting: Loading dependencies and libraries...")
import time
_start_time = time.time()
import torch
import yaml
import numpy as np
from transformers import RobertaTokenizer
from pathlib import Path
import sys, os
import matplotlib.pyplot as plt
import seaborn as sns
import re as _re, unicodedata as _ud, html as _html

# ── Path resolution ────────────────────────────────────────────────────────────
ML_RESEARCH = os.path.dirname(os.path.abspath(''))
os.chdir(ML_RESEARCH)  # ml-research/
SRC_DIR = os.path.join(ML_RESEARCH, 'src')
if SRC_DIR not in sys.path: sys.path.insert(0, SRC_DIR)

from models.model import create_model
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Loading dependencies and libraries.")


⏳ Starting: Loading dependencies and libraries...
🕒 Done in 6.48s
✅ Completed: Loading dependencies and libraries.


## Inference-Time Text Cleaning
Must match the preprocessing pipeline in `02_preprocess_and_split.ipynb` exactly — otherwise the model encounters distribution shift.

In [2]:
print("⏳ Starting: Defining function _is_garbled...")
import time
_start_time = time.time()
_CONSONANTS  = set('bcdfghjklmnpqrstvwxz')
_HTML_RE     = _re.compile(r'<[^>]+>')
_URL_RE      = _re.compile(r'https?://\S+|www\.\S+', _re.IGNORECASE)
_EMAIL_RE    = _re.compile(r'[\w.+-]+@[\w-]+\.[a-zA-Z]{2,}', _re.IGNORECASE)

def _is_garbled(tok: str, min_len=6, cons_ratio=0.82, rep_ratio=0.60) -> bool:
    """Same garbled-token heuristic as preprocessing."""
    t = tok.lower()
    if len(t) < min_len: return False
    letters = [c for c in t if c.isalpha()]
    if not letters: return False
    return (sum(c in _CONSONANTS for c in letters) / len(letters) >= cons_ratio
            or max(t.count(c) for c in set(t)) / len(t) >= rep_ratio)

def clean_text_for_inference(text: str) -> str:
    """Clean user-submitted review text to match training distribution."""
    if not isinstance(text, str) or not text.strip(): return ''
    text = _ud.normalize('NFC', _html.unescape(text))  # Unicode normalisation + HTML decode
    text = _HTML_RE.sub(' ', text)                      # Strip remaining HTML tags
    text = _URL_RE.sub(' ', text)                       # Remove URLs
    text = _EMAIL_RE.sub(' ', text)                     # Remove emails
    # Normalise repeated punctuation (translation artifact)
    text = _re.sub(r'\.{3,}', '…', text)
    text = _re.sub(r'!{2,}', '!', text)
    text = _re.sub(r'\?{2,}', '?', text)
    text = _re.sub(r'[\u200b-\u200f\u202a-\u202e\ufeff]', '', text)  # Zero-width chars

    tokens      = text.split()
    clean_toks  = [t for t in tokens if not _is_garbled(t)]
    # Drop heavily garbled text entirely (>40% of tokens garbled)
    if len(tokens) >= 5 and len(tokens) - len(clean_toks) >= 0.4 * len(tokens):
        return ''

    return _re.sub(r' {2,}', ' ', ' '.join(clean_toks)).strip()
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Defining function _is_garbled.")


⏳ Starting: Defining function _is_garbled...
🕒 Done in 0.00s
✅ Completed: Defining function _is_garbled.


## SentimentPredictor
The main production predictor class. Loads checkpoint, provides `predict()`, and several explanation methods.

In [3]:
print("⏳ Starting: Defining class SentimentPredictor...")
import time
_start_time = time.time()
class SentimentPredictor:
    """
    Wraps the trained MultiAspectSentimentModel for single-review inference.

    Args:
        checkpoint_path: Path to best_model.pt
        device:          'cuda' or 'cpu' (auto-detected)
        temperature:     Softmax temperature for calibration (0.5 works well for this model)
    """
    def __init__(self, checkpoint_path, device='cuda', temperature=0.5):
        self.device      = torch.device(device if torch.cuda.is_available() else 'cpu')
        self.temperature = temperature

        print("📥 Loading a saved model checkpoint...")
        checkpoint = torch.load(checkpoint_path, map_location=self.device)
        self.config = checkpoint['config']

        # Recreate the model from config + load weights
        print("🧩 Building the neural network architecture...")
        self.model = create_model(self.config)
        self.model.load_state_dict(checkpoint['model_state_dict'])
        self.model.to(self.device).eval()  # Set to evaluation mode (disables dropout)

        self.tokenizer   = RobertaTokenizer.from_pretrained(self.config['model']['roberta_model'])
        self.aspect_names = self.config['aspects']['names']
        self.aspect_to_id = {n: i for i, n in enumerate(self.aspect_names)}
        self.label_names  = ['negative', 'neutral', 'positive']

    def predict(self, text: str, aspect: str, return_attention=False) -> dict:
        """
        Predict the sentiment of `text` for the given `aspect`.

        Args:
            text:             Raw user review text (will be cleaned internally)
            aspect:           Aspect name (e.g. 'colour', 'smell')
            return_attention: If True, also return token attention weights

        Returns:
            dict with keys: aspect, sentiment, confidence, probabilities, top_tokens
            + 'attention' key if return_attention=True
        """
        if aspect not in self.aspect_to_id:
            raise ValueError(f'Invalid aspect: {aspect}. Choose from {self.aspect_names}')

        text = clean_text_for_inference(text)
        if not text:
            # Empty text after cleaning → return flat/uniform probabilities
            return {'aspect': aspect, 'sentiment': 'neutral', 'confidence': 0.0,
                    'probabilities': {'negative': .33, 'neutral': .34, 'positive': .33}}

        encoding = self.tokenizer(
            text, add_special_tokens=True,
            max_length=self.config['data']['max_seq_length'],
            padding='max_length', truncation=True, return_tensors='pt'
        )
        input_ids      = encoding['input_ids'].to(self.device)
        attention_mask = encoding['attention_mask'].to(self.device)
        aspect_id      = torch.tensor([self.aspect_to_id[aspect]], dtype=torch.long, device=self.device)

        # CRITICAL: Pass empty edge_index list to force GCN branch (uses final_classifier).
        # Without this, model.forward() bypasses the final_classifier layer.
        empty_edge_index = [torch.zeros(2, 0, dtype=torch.long, device=self.device)]

        with torch.no_grad():
            if return_attention:
                logits, attn_weights, _, _ = self.model(
                    input_ids, attention_mask, aspect_id,
                    edge_index=empty_edge_index, return_attention=True
                )
            else:
                logits = self.model(input_ids, attention_mask, aspect_id, edge_index=empty_edge_index)

        # Temperature scaling: amplifies logit differences to produce confident predictions
        probs      = torch.softmax(logits / self.temperature, dim=1)[0]
        pred_class = torch.argmax(probs).item()

        result = {
            'aspect'        : aspect,
            'sentiment'     : self.label_names[pred_class],
            'confidence'    : probs[pred_class].item(),
            'probabilities' : {
                'negative': probs[0].item(),
                'neutral' : probs[1].item(),
                'positive': probs[2].item(),
            },
        }

        if return_attention:
            tokens         = self.tokenizer.convert_ids_to_tokens(input_ids[0])
            valid_len      = attention_mask[0].sum().item()
            tokens         = tokens[:valid_len]
            attn           = attn_weights[0].cpu().numpy()[:valid_len]

            result['attention'] = {'tokens': tokens, 'weights': attn.tolist()}

            # Extract top-3 meaningful words (non-special, non-stopword, alphabetic)
            SPECIAL   = {'<s>', '</s>', '<pad>', '<mask>'}
            STOPWORDS = {'the', 'a', 'an', 'is', 'it', 'i', 'this', 'and', 'or', 'but',
                         'to', 'of', 'in', 'for', 'with', 'my', 'was', 'are', 'be'}
            valid_idx  = [i for i, t in enumerate(tokens) if t not in SPECIAL]
            if valid_idx:
                top_idx  = sorted(valid_idx, key=lambda i: attn[i], reverse=True)
                top_toks = [tokens[i].lstrip('Ġ▁').strip() for i in top_idx]
                top_toks = [t for t in top_toks if t.isalpha() and len(t) > 2 and t.lower() not in STOPWORDS]
                seen, unique = set(), []
                for t in top_toks:
                    if t.lower() not in seen: seen.add(t.lower()); unique.append(t)
                result['top_tokens'] = unique[:3]
            else:
                result['top_tokens'] = []
        else:
            result['top_tokens'] = []

        return result

    def predict_all_aspects(self, text: str) -> dict:
        """Run predict() for every aspect. Returns {aspect: prediction_dict}."""
        return {asp: self.predict(text, asp) for asp in self.aspect_names}
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Defining class SentimentPredictor.")


⏳ Starting: Defining class SentimentPredictor...
🕒 Done in 0.00s
✅ Completed: Defining class SentimentPredictor.


## Quick Inference Demo

In [4]:
print("⏳ Starting: Loading trained model...")
import time
_start_time = time.time()
# ── Uncomment the block below to run interactively after model training ────────

# import os
# ckpt = os.path.join(os.getcwd(), 'outputs', 'cosmetic_sentiment_v1', 'best_model.pt')
# predictor = SentimentPredictor(ckpt)
#
# text = 'I love the vibrant colour and the smell is amazing, but shipping took 3 weeks.'
# for asp in predictor.aspect_names:
#     res = predictor.predict(text, asp)
#     if res['confidence'] > 0.25:
#         print(f"{asp:16s}: {res['sentiment']:8s}  conf={res['confidence']:.2%}")

print('SentimentPredictor loaded. Instantiate with checkpoint path to run inference.')
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Loading trained model.")


⏳ Starting: Loading trained model...
SentimentPredictor loaded. Instantiate with checkpoint path to run inference.
🕒 Done in 0.00s
✅ Completed: Loading trained model.
